# 🏋️ Fun with Text Embeddings 🍎

Welcome to our **Health & Fitness** embeddings notebook! In this tutorial, we'll show you how to:

1. **Initialize** an `AIProjectClient` to access your Azure AI Foundry project.
2. **Embed text** using `azure-ai-inference` with our fun health-themed phrases.
3. **Use a prompt template** for extra context.

Let's get started and have some fun with our healthy ideas! 🍏

## 1. Setup & Environment

#### Prerequisites:
- Deploy a text embeddings model (**text-embedding-3-small**) in Azure AI Foundry

#
We'll import our libraries and load the environment variables for:
- `PROJECT_CONNECTION_STRING`: Your project connection string.
- `TEXT_EMBEDDING_MODEL`: The text embeddings model deployment name.

We'll import libraries, load environment variables, and create an `AIProjectClient`.

Let's begin! 🚀

In [ ]:
import os
import json
from pathlib import Path

def find_cred_json(start_path):
    current = Path(start_path)
    while current != current.parent:
        cred_file = current / 'cred.json'
        if cred_file.exists():
            return str(cred_file)
        current = current.parent
    return None

start_dir = os.getcwd()  # Use current directory
file_path = find_cred_json(start_dir)

print(f"Found cred.json at: {file_path}")

try:
    with open(file_path, 'r') as f:
        loaded_config = json.load(f)
    
    print("Project Connection String:", loaded_config['PROJECT_CONNECTION_STRING'])
    print("Tenant ID:", loaded_config['TENANT_ID'])
    print("Model Deployment ID:", loaded_config['MODEL_DEPLOYMENT_NAME'])

except FileNotFoundError:
    print(f"Could not find file at: {file_path}")
except json.JSONDecodeError:
    print(f"File exists but contains invalid JSON")
except TypeError:
    print("File path was None — cred.json not found in any parent directories.")


In [ ]:
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Retrieve from environment or fallback
PROJECT_CONNECTION_STRING = loaded_config.get("PROJECT_CONNECTION_STRING", "<your-connection-string>")
TEXT_EMBEDDING_MODEL = loaded_config.get("TEXT_EMBEDDING_MODEL", "text-embedding-3-small")

# Initialize AI Project Client
project_client = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint=loaded_config["PROJECT_ENDPOINT"],
)

# Get Azure OpenAI client
openai_client = project_client.get_openai_client(
    api_version="2024-10-21"  # Match your Azure OpenAI API version
)

## 2. Text Embeddings

We'll call `embeddings` from our `openai_client` to retrieve the embeddings client. Then we'll embed some fun health-themed phrases:

- "🍎 An apple a day keeps the doctor away"
- "🏋️ 15-minute HIIT workout routine"
- "🧘 Mindful breathing exercises"

The output will be numeric vectors representing each phrase in semantic space. Let’s see those embeddings!

In [ ]:
# Text to embed
text_phrases = [
    "An apple a day keeps the doctor away 🍎",
    "Quick 15-minute HIIT workout routine 🏋️",
    "Mindful breathing exercises 🧘"
]

try:
    # Call embeddings API
    response = openai_client.embeddings.create(
        model=TEXT_EMBEDDING_MODEL,  # Set model name in env vars
        input=text_phrases
    )

    # Loop through embeddings
    for item in response.data:
        vec = item.embedding
        sample_str = f"[{vec[0]:.4f}, {vec[1]:.4f}, ..., {vec[-2]:.4f}, {vec[-1]:.4f}]"
        print(
            f"Sentence {item.index}: '{text_phrases[item.index]}':\n"
            f"  Embedding length={len(vec)}\n"
            f"  Sample: {sample_str}\n"
        )

except Exception as e:
    print("❌ Error embedding text:", e)


## 3. Prompt Template Example 📝

Even though our focus is on embeddings, here's how you might prepend some context to a user message. Imagine you want to embed user text but first add a system prompt such as “You are HealthFitGPT, a fitness guidance model…” This little extra helps set the stage for more context-aware embeddings.


In [ ]:
# A basic prompt template (system-style) we'll prepend to user text
TEMPLATE_SYSTEM = (
    "You are HealthFitGPT, a fitness guidance model.\n"
    "Please focus on healthy advice and disclaim you're not a doctor.\n\n"
    "User message:"  # We'll append the user message after this
)

def embed_with_template(user_text: str):
    """Embed user text with a system template in front."""
    content = TEMPLATE_SYSTEM + " " + user_text

    rsp = openai_client.embeddings.create(
        model= TEXT_EMBEDDING_MODEL,
        input=[content]
    )

    return rsp.data[0].embedding

# Example usage
sample_user_text = "Can you suggest a quick home workout for busy moms?"
embedding_result = embed_with_template(sample_user_text)

print("Embedding length:", len(embedding_result))
print("First few dims:", embedding_result[:8])


## 6. Wrap-Up & Next Steps
🎉 We've shown how to:
- Set up the `AIProjectClient`.
- Get **text embeddings** using *text-embedding-3-small*.
- Use a **prompt template** to add system context to your embeddings.

Have fun experimenting, and remember: when it comes to your health, always consult a professional!